# Day 17 — Agents and Tool Use
## 30 Days of AI: From NLP to LLMs

---

Every system you have built so far follows the same pattern:
input → LLM → output. The LLM responds once and the pipeline ends.

Agents break this pattern. An agent is an LLM that can decide
WHAT to do, DO it (use tools), OBSERVE the result, and then
decide what to do NEXT — in a loop until the task is complete.

This is the shift from language model to autonomous system.
An agent with the right tools can search the web, run code,
query databases, call APIs, and compose multi-step workflows
that no single LLM call could accomplish.

---

### What You Will Learn Today

- The ReAct framework — Reason + Act in an interleaved loop
- Tools — wrapping any function as an LLM-callable action
- Function calling — how OpenAI/Anthropic APIs enable tool use
- Building tools from scratch: calculator, search, document lookup
- LangChain AgentExecutor — the ReAct loop implemented
- Agent memory and stopping conditions
- Multi-tool agents — agents that choose among many tools
- Common failure modes: loops, hallucinated tool calls, infinite loops

### Goal by End of Day

Build a multi-tool agent that can answer complex questions by
combining calculation, document search, and text analysis —
choosing the right tool for each step autonomously.

In [ ]:
## Run once
## !pip install langchain langchain-openai langchain-anthropic \
##             langchain-community sentence-transformers faiss-cpu -q

import os
import re
import json
import math
import warnings
warnings.filterwarnings('ignore')

HAS_OPENAI    = bool(os.environ.get('OPENAI_API_KEY'))
HAS_ANTHROPIC = bool(os.environ.get('ANTHROPIC_API_KEY'))

def get_llm(temperature=0.0):
    if HAS_OPENAI:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model='gpt-3.5-turbo', temperature=temperature)
    elif HAS_ANTHROPIC:
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model='claude-3-haiku-20240307', temperature=temperature)
    else:
        from langchain_core.language_models.fake import FakeListChatModel
        # Fake responses simulate ReAct reasoning traces
        return FakeListChatModel(responses=[
            'Thought: I need to calculate this.\nAction: calculator\nAction Input: 2500 * 0.15',
            'Thought: I have the answer.\nFinal Answer: The result is 375.',
            'Thought: I should search for this.\nAction: search_docs\nAction Input: overfitting prevention',
            'Thought: Found the answer.\nFinal Answer: Use regularization and dropout.',
        ] * 10)

provider = 'openai' if HAS_OPENAI else 'anthropic' if HAS_ANTHROPIC else 'mock'
print(f'Provider: {provider}')
print('Ready.')

---

## Part 1 — The ReAct Framework

ReAct (Reason + Act) was introduced by Yao et al. (2022). It is the
most widely used pattern for building LLM agents.

```
Standard LLM call:
  Input → LLM → Output   (one shot)

ReAct Agent loop:
  Input
    │
    ▼
  Thought  : LLM reasons about what to do next
    │
    ▼
  Action   : LLM selects a tool and provides input
    │
    ▼
  Observation : Tool runs, result returned to LLM
    │
    └──────────────────────────────────────┐
    ▼                                      │
  Thought  : LLM reasons about observation │
    │                                      │
    ▼                                      │
  Action OR Final Answer  ─────────────────┘
                             (loop until Final Answer)

Concrete example:
  Question : 'What is 15% of the average of 240 and 360?'

  Thought  : I need to find the average of 240 and 360 first.
  Action   : calculator
  Input    : (240 + 360) / 2
  Observe  : 300

  Thought  : Now I need 15% of 300.
  Action   : calculator
  Input    : 300 * 0.15
  Observe  : 45.0

  Thought  : I have the final answer.
  Final Answer: 45.0
```

In [ ]:
# ----------------------------------------------------------------
# Part 2 — Building Tools from scratch
# A Tool is any Python function wrapped with a description.
# The description is what the LLM reads to decide when to use it.
# ----------------------------------------------------------------

from langchain_core.tools import tool

# Tool 1: Safe calculator
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use for arithmetic, percentages, and numerical calculations.
    Input: a valid Python math expression like '2500 * 0.15' or 'math.sqrt(144)'.
    Do NOT use for anything other than pure math expressions.
    """
    try:
        # Safe eval: only allow math operations
        allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
        allowed.update({'abs': abs, 'round': round, 'min': min, 'max': max})
        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(round(float(result), 6))
    except Exception as e:
        return f'Error: {str(e)}'


# Tool 2: Knowledge base search
# (uses a small in-memory corpus)
KNOWLEDGE_BASE = {
    'overfitting': 'Overfitting occurs when a model learns training data too well, '
                   'including noise. Prevention: regularization, dropout, early stopping, '
                   'cross-validation, more training data.',
    'transformer': 'The Transformer uses self-attention to process sequences in parallel. '
                   'Key components: multi-head attention, positional encoding, '
                   'feed-forward layers, residual connections.',
    'rag':         'RAG combines retrieval with generation. Documents are chunked, embedded, '
                   'and stored in a vector store. Relevant chunks are injected into the '
                   'LLM prompt to ground the answer.',
    'rlhf':        'RLHF (Reinforcement Learning from Human Feedback) aligns LLMs with '
                   'human preferences. Stages: supervised fine-tuning, reward model training, '
                   'PPO optimization against the reward model.',
    'attention':   'Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V. '
                   'Q=query, K=key, V=value. Scaled dot-product attention.',
    'bert':        'BERT is an encoder-only Transformer pretrained with masked language '
                   'modeling and next sentence prediction. 110M params (base), 340M (large).',
    'fine-tuning': 'Fine-tuning adapts a pretrained model to a specific task. '
                   'Full fine-tuning updates all weights. LoRA trains low-rank adapters only.',
    'lora':        'LoRA (Low-Rank Adaptation) freezes the original weights and adds small '
                   'trainable matrices A and B. ΔW = B×A. Reduces trainable params by 99%.',
}

@tool
def search_knowledge_base(query: str) -> str:
    """
    Searches the AI/ML knowledge base for information.
    Use for questions about machine learning concepts, models, and techniques.
    Input: a keyword or short phrase describing what you want to know.
    Returns relevant information from the knowledge base.
    """
    query_lower = query.lower()
    # Simple keyword matching
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower:
            return f'Found: {value}'
    # Fuzzy: check if any word in query matches any key
    words = set(query_lower.split())
    for key, value in KNOWLEDGE_BASE.items():
        if any(w in key or key in w for w in words):
            return f'Found (partial match on "{key}"): {value}'
    return 'No information found. Try different keywords.'


# Tool 3: Text analyzer
@tool
def analyze_text(text: str) -> str:
    """
    Analyzes a text and returns statistics: word count, sentence count,
    average sentence length, and most frequent words.
    Use when asked to analyze or describe properties of a piece of text.
    Input: the text to analyze.
    """
    words     = text.lower().split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    # Word frequency (exclude stopwords)
    stopwords = {'the','a','an','is','in','it','of','to','and','or','for','with','that'}
    freq      = {}
    for w in words:
        w = w.strip('.,!?;:"\'')
        if w and w not in stopwords:
            freq[w] = freq.get(w, 0) + 1
    top5 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:5]
    return (
        f'Word count: {len(words)}, '
        f'Sentences: {len(sentences)}, '
        f'Avg sentence length: {len(words)/max(len(sentences),1):.1f} words, '
        f'Top words: {[(w,c) for w,c in top5]}'
    )


# Tool 4: Unit converter
@tool
def unit_converter(query: str) -> str:
    """
    Converts between common units.
    Supports: km/miles, kg/lbs, celsius/fahrenheit, MB/GB.
    Input format: '<value> <from_unit> to <to_unit>'. Example: '100 km to miles'.
    """
    conversions = {
        ('km',   'miles') : lambda x: x * 0.621371,
        ('miles','km')    : lambda x: x * 1.60934,
        ('kg',   'lbs')   : lambda x: x * 2.20462,
        ('lbs',  'kg')    : lambda x: x / 2.20462,
        ('c',    'f')     : lambda x: x * 9/5 + 32,
        ('f',    'c')     : lambda x: (x - 32) * 5/9,
        ('mb',   'gb')    : lambda x: x / 1024,
        ('gb',   'mb')    : lambda x: x * 1024,
    }
    try:
        parts   = query.lower().strip().split()
        value   = float(parts[0])
        from_u  = parts[1].lower()
        to_u    = parts[3].lower()
        key     = (from_u, to_u)
        if key in conversions:
            result = conversions[key](value)
            return f'{value} {from_u} = {result:.4f} {to_u}'
        return f'Conversion {from_u} → {to_u} not supported.'
    except Exception as e:
        return f'Parse error: {str(e)}. Format: "<value> <unit> to <unit>"'


# Test tools independently first
print('Tool Tests')
print('=' * 55)
print('calculator("(240 + 360) / 2")             :', calculator.invoke('(240 + 360) / 2'))
print('calculator("300 * 0.15")                   :', calculator.invoke('300 * 0.15'))
print('search_knowledge_base("overfitting")        :', search_knowledge_base.invoke('overfitting')[:80] + '...')
print('unit_converter("100 km to miles")           :', unit_converter.invoke('100 km to miles'))
print('analyze_text("Hello world. This is a test."):', analyze_text.invoke('Hello world. This is a test.'))

---

## Part 3 — Function Calling vs ReAct

```
There are two ways to give an LLM access to tools:

ReAct (text-based):
  → The LLM outputs structured text: 'Action: tool_name\nAction Input: ...'
  → The framework parses this text and calls the tool
  → Works with any LLM, including local models
  → Fragile: depends on model following exact output format
  → Thought traces are visible in output

Function Calling / Tool Use (API feature):
  → The API has a 'tools' parameter with JSON schemas for each tool
  → The LLM outputs a structured JSON tool call (not free text)
  → The framework executes the tool and passes result back
  → More reliable: structured JSON, not text parsing
  → Supported by: OpenAI, Anthropic, Google, Mistral

LangChain abstracts both patterns behind the same interface.
When you use .bind_tools() the right approach is selected
automatically based on the model provider.
```

In [ ]:
# ----------------------------------------------------------------
# Part 4 — ReAct Agent with LangChain AgentExecutor
# ----------------------------------------------------------------

from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# All available tools
tools = [calculator, search_knowledge_base, analyze_text, unit_converter]

# The ReAct prompt template from LangChain Hub
# It structures the Thought/Action/Observation loop
try:
    react_prompt = hub.pull('hwchase17/react')
    print('ReAct prompt loaded from Hub.')
except:
    # Fallback prompt if Hub is not accessible
    from langchain_core.prompts import PromptTemplate
    react_prompt = PromptTemplate.from_template("""
Answer the following question as best you can. You have access to these tools:

{tools}

Use the following format EXACTLY:

Question: the input question you must answer
Thought: think about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")
    print('Using fallback ReAct prompt.')

# Create agent and executor
agent = create_react_agent(
    llm    = get_llm(temperature=0),
    tools  = tools,
    prompt = react_prompt,
)

agent_executor = AgentExecutor(
    agent          = agent,
    tools          = tools,
    verbose        = True,    # show Thought/Action/Observation traces
    max_iterations = 8,       # safety limit — prevents infinite loops
    handle_parsing_errors = True,
)

print('\nAgent executor ready.')
print(f'Tools available: {[t.name for t in tools]}')

In [ ]:
# ----------------------------------------------------------------
# Run the agent on multi-step questions
# Watch the Thought/Action/Observation trace
# ----------------------------------------------------------------

questions = [
    # Requires calculator (multi-step)
    'What is 15% of the average of 240 and 360?',

    # Requires knowledge base search
    'What is LoRA and how does it reduce GPU memory usage?',

    # Requires calculator + unit conversion
    'I trained my model for 2000 steps. Each step processes 32 samples. '
    'How many total samples were processed? Also convert 180 lbs to kg.',
]

for question in questions:
    print('\n' + '='*65)
    print(f'QUESTION: {question}')
    print('='*65)
    try:
        result = agent_executor.invoke({'input': question})
        print(f'\nFINAL ANSWER: {result["output"]}')
    except Exception as e:
        print(f'Agent error: {e}')

In [ ]:
# ----------------------------------------------------------------
# Part 5 — Tool Calling API (structured, more reliable)
# Uses model's native function calling instead of text parsing
# ----------------------------------------------------------------

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.runnables import RunnableLambda

# Bind tools to the LLM
# .bind_tools() tells the model about available tools via the API
llm_with_tools = get_llm(temperature=0).bind_tools(tools)

def run_tool(tool_call):
    """Execute a tool call returned by the LLM."""
    tool_map = {t.name: t for t in tools}
    tool_name = tool_call['name']
    tool_args = tool_call['args']

    if tool_name not in tool_map:
        return f'Unknown tool: {tool_name}'

    # Get the first argument value for tools that take a single string
    if isinstance(tool_args, dict) and len(tool_args) == 1:
        arg_value = list(tool_args.values())[0]
    else:
        arg_value = str(tool_args)

    return tool_map[tool_name].invoke(arg_value)


def tool_calling_agent(question: str, max_steps: int = 6) -> str:
    """
    Simple tool-calling agent loop.
    Mirrors what AgentExecutor does internally.

    1. Send messages to LLM
    2. If LLM returns tool_calls → execute tools, add results
    3. Send updated messages back → repeat
    4. When no tool_calls → return final text response
    """
    messages = [HumanMessage(content=question)]
    step = 0

    print(f'Question: {question}')
    print('-' * 55)

    while step < max_steps:
        step += 1
        response = llm_with_tools.invoke(messages)

        # Check if model wants to call tools
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tc in response.tool_calls:
                print(f'  Step {step}: calling {tc["name"]}({tc["args"]})')
                result = run_tool(tc)
                print(f'           → {result}')
                # Append tool result to conversation
                messages.append(response)
                messages.append(ToolMessage(
                    content   = str(result),
                    tool_call_id = tc['id'],
                ))
        else:
            # No tool calls → final answer
            print(f'\nFinal Answer: {response.content}')
            return response.content

    return 'Max steps reached without final answer.'


print('Tool-Calling Agent (structured function calling)')
print('=' * 55)
try:
    tool_calling_agent(
        'What is 18% of 2500? Also, what does RLHF stand for and what does it do?'
    )
except Exception as e:
    print(f'Note: Tool calling requires a real API key: {type(e).__name__}')
    print('The agent still works with AgentExecutor in the cell above.')

In [ ]:
# ----------------------------------------------------------------
# Part 6 — Agent with Memory
# Combining agent tool use with conversation history
# ----------------------------------------------------------------

from langchain.memory import ConversationBufferMemory
from langchain.agents import create_react_agent, AgentExecutor

# Memory stores the conversation so the agent can reference
# earlier turns when answering follow-up questions

agent_memory = ConversationBufferMemory(
    memory_key      = 'chat_history',
    return_messages = True,
)

# Use a conversational ReAct prompt that includes history
try:
    conv_react_prompt = hub.pull('hwchase17/react-chat')
except:
    # Fallback
    from langchain_core.prompts import PromptTemplate
    conv_react_prompt = PromptTemplate.from_template("""
Assistant is a helpful AI with access to tools.

Tools: {tools}
Previous conversation: {chat_history}

Question: {input}
Thought:{agent_scratchpad}""")

conv_agent = create_react_agent(
    llm    = get_llm(temperature=0.1),
    tools  = tools,
    prompt = conv_react_prompt,
)

conv_executor = AgentExecutor(
    agent              = conv_agent,
    tools              = tools,
    memory             = agent_memory,
    verbose            = False,
    max_iterations     = 6,
    handle_parsing_errors = True,
)

# Simulated conversation with follow-ups
conversation = [
    'My model processes 32 samples per batch. How many samples in 500 batches?',
    'What percentage is that of 1 million samples?',
    'Remind me: what was my batch size?',   # tests memory
]

print('Conversational Agent with Memory')
print('=' * 55)

for turn, question in enumerate(conversation, 1):
    print(f'\nTurn {turn}: {question}')
    try:
        result = conv_executor.invoke({'input': question})
        print(f'Answer: {result["output"]}')
    except Exception as e:
        print(f'(Requires API key: {type(e).__name__})')

In [ ]:
# ----------------------------------------------------------------
# Part 7 — Build your own ReAct loop from scratch
# No framework dependency — understand every line
# ----------------------------------------------------------------

import os

def simple_react_loop(question: str, tools: list, max_steps=6):
    """
    Minimal ReAct loop implementation.
    Shows exactly what frameworks like LangChain do internally.

    Steps:
      1. Build prompt with question + tool descriptions + scratchpad
      2. Call LLM
      3. Parse Action/Action Input from response
      4. Execute the tool
      5. Append Observation to scratchpad
      6. Repeat until 'Final Answer:' is found
    """

    # Build tool descriptions
    tool_map = {t.name: t for t in tools}
    tool_desc = '\n'.join(
        f'{t.name}: {t.description.strip()[:100]}'
        for t in tools
    )
    tool_names = ', '.join(tool_map.keys())

    REACT_TEMPLATE = """
Answer the following question using the available tools.
Available tools:
{tool_desc}

Format:
Thought: <reasoning>
Action: <tool_name>
Action Input: <input>
Observation: <tool result — filled by system>
... repeat as needed ...
Thought: I have the final answer
Final Answer: <answer>

Question: {question}
{scratchpad}"""

    scratchpad = ''

    # Import call_llm from earlier or use a simple wrapper
    def _call(prompt):
        if HAS_OPENAI:
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model='gpt-3.5-turbo',
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0, max_tokens=300,
            )
            return resp.choices[0].message.content
        elif HAS_ANTHROPIC:
            import anthropic
            client = anthropic.Anthropic()
            resp = client.messages.create(
                model='claude-3-haiku-20240307', max_tokens=300,
                messages=[{'role': 'user', 'content': prompt}]
            )
            return resp.content[0].text
        else:
            return 'Thought: I need to calculate this.\nAction: calculator\nAction Input: 2000 * 32\nFinal Answer: 64000 samples total.'

    print(f'Question: {question}')
    print('─' * 55)

    for step in range(max_steps):
        prompt   = REACT_TEMPLATE.format(
            tool_desc  = tool_desc,
            question   = question,
            scratchpad = scratchpad,
        )
        response = _call(prompt)

        # Append response to scratchpad
        scratchpad += response

        # Check for final answer
        if 'Final Answer:' in response:
            final = response.split('Final Answer:')[-1].strip()
            print(f'Final Answer: {final}')
            return final

        # Parse Action and Action Input
        action_match = re.search(r'Action:\s*(.+?)\n', response)
        input_match  = re.search(r'Action Input:\s*(.+?)(?:\n|$)', response)

        if action_match and input_match:
            action = action_match.group(1).strip()
            ainput = input_match.group(1).strip()

            print(f'  Step {step+1}: {action}({ainput})')

            if action in tool_map:
                result = tool_map[action].invoke(ainput)
            else:
                result = f'Unknown tool: {action}'

            print(f'           → {result}')
            scratchpad += f'\nObservation: {result}\n'
        else:
            print('No action found in response — agent may have answered directly.')
            print(response[:200])
            return response

    return 'Max steps reached.'


print('Custom ReAct Loop (from scratch, no LangChain)')
print('=' * 55)
simple_react_loop(
    'How many samples are processed in 2000 batches of 32? '
    'Also convert the result to GB assuming each sample is 4 bytes.',
    tools=[calculator, unit_converter]
)

---

## Part 8 — Agent Failure Modes and Mitigations

```
Failure 1: Infinite loops
  Symptom  : Agent keeps calling tools, never reaches Final Answer
  Cause    : Tool results do not satisfy the agent's reasoning
  Fix      : Set max_iterations (e.g. 8). Log each step.
             Return early_stopping_method='generate' to force answer.

Failure 2: Hallucinated tool inputs
  Symptom  : Agent calls calculator with 'what is 15% of revenue'
             instead of a numeric expression
  Cause    : LLM did not extract the number before calling the tool
  Fix      : Add input validation inside the tool function.
             Better tool description with explicit input format.

Failure 3: Wrong tool selected
  Symptom  : Agent uses calculator for a factual question
  Cause    : Tool descriptions are ambiguous or overlapping
  Fix      : Make tool descriptions very specific about when NOT to use.
             'Do NOT use this for...'

Failure 4: Ignoring tool results
  Symptom  : Agent calls a tool but uses its own answer anyway
  Cause    : Model confidence overrides tool observation
  Fix      : Add to system prompt: 'Always use the Observation result.
             Never override tool output with your own calculation.'

Failure 5: Tool errors not handled
  Symptom  : Tool raises exception, agent crashes
  Fix      : Always wrap tool logic in try/except.
             Return descriptive error messages.
             Set handle_parsing_errors=True in AgentExecutor.
```

---

## Day 17 Summary

```
What you built today:

1.  @tool decorator         →  wrap any function as an LLM-callable tool
2.  4 tools                 →  calculator, knowledge_base, text_analyzer, converter
3.  ReAct agent             →  create_react_agent + AgentExecutor
4.  Tool-calling agent      →  structured JSON function calling loop
5.  Conversational agent    →  agent + ConversationBufferMemory
6.  Custom ReAct loop       →  from scratch, no framework dependency
7.  Failure modes + fixes   →  loops, hallucinated inputs, wrong tools

Key mental model:
  An agent is an LLM in a loop.
  Tools are the actions it can take.
  The loop runs until Final Answer or max_iterations.
  Everything else (memory, chains, RAG) can be a tool.

What comes next:
  Day 18 — Capstone Project: build a complete RAG Agent that
  combines everything from Days 11-17 into one production-ready
  document Q&A system with tools, memory, and evaluation.
```

### Self-Check Questions

1. What is the difference between ReAct (text parsing) and
   function calling (structured JSON)? When would you use each?
2. Why does `max_iterations` exist? What happens without it?
3. Your agent keeps calling the calculator with the string
   'the answer'. How do you fix this?
4. How is an agent with a search_knowledge_base tool
   different from a standalone RAG pipeline?
5. What does `handle_parsing_errors=True` do in AgentExecutor?